In [1]:
import os
from dotenv import load_dotenv
from pathlib import Path

import json
import time
from urllib.parse import urljoin, urlencode
import requests

import numpy as np
import pandas as pd

import plotly
import plotly.graph_objs as go
import plotly.express as px
import seaborn as sns

In [2]:
env_path = Path("/home/jovyan/.env")
load_dotenv(env_path)
FRED_API_KEY = os.getenv("FRED_API_KEY")
J_QUONTS_MAILADDRESS = os.getenv("J_QUONTS_MAILADDRESS")
J_QUONTS_PASSWORD = os.getenv("J_QUONTS_PASSWORD")

In [ ]:
current_dir = Path.cwd()
data_path = (current_dir / "data").resolve()

In [4]:
print(data_path)
bond_boj = pd.read_csv(
    data_path / "nme_R031.3169477.20260329112653.01.csv",
    encoding="utf-8",
    header=1, 
).rename(columns={"系列名称": "年", "__種類別内訳＿種類別内訳/国債/内国債": "国債残高"})
bond_boj.tail()

/home/jovyan/data


,年,＿種類別内訳/国債/内国債
20,2020,9838777
21,2021,10237117
22,2022,10626907
23,2023,10975593
24,2024,11325480


In [5]:
# mof_url = 'https://www.mof.go.jp/jgbs/reference/interest_rate/'
# csv_url = urljoin(mof_url, 'data/jgbcm.csv')
# all_csv_url = urljoin(mof_url, 'data/jgbcm_all.csv')
jgbcm_mof = pd.read_csv(
    data_path / "jgbcm.csv",
    encoding="utf-8",
    skiprows=1,
    index_col=0,
    na_values="-",
)
jgbcm_all = pd.read_csv(
    data_path / "jgbcm_all.csv",
    encoding="utf-8",
    skiprows=1,
    index_col=0,
    na_values="-",
)
jgbcm_concatenated = pd.concat([jgbcm_all, jgbcm_mof]).dropna(axis="index", how="all")
jgbcm_concatenated.index[:5], jgbcm_concatenated.index[5000:5005], jgbcm_concatenated.index[-5:]


(Index(['S49.9.24', 'S49.9.25', 'S49.9.26', 'S49.9.27', 'S49.9.28'], dtype='object', name='基準日'),
 Index(['H4.11.16', 'H4.11.17', 'H4.11.18', 'H4.11.19', 'H4.11.20'], dtype='object', name='基準日'),
 Index(['R8.3.19', 'R8.3.23', 'R8.3.24', 'R8.3.25', 'R8.3.26'], dtype='object', name='基準日'))

In [6]:
era_offsets = {"S": 1925, "H": 1988, "R": 2018}
ref_date_df = jgbcm_concatenated.index.to_series().str.extract(
    r"(?P<era>[SHR])(?P<year>\d+)\.(?P<month>\d+)\.(?P<day>\d+)",
    expand=True,
)
ce_index = pd.to_datetime({
    "year": ref_date_df["year"].astype(int) + ref_date_df["era"].map(era_offsets),
    "month": ref_date_df["month"].astype(int),
    "day": ref_date_df["day"].astype(int),
})
ce_index.name = jgbcm_concatenated.index.name
interest_rate = jgbcm_concatenated.copy().set_index(ce_index)
interest_rate

,1年,2年,3年,4年,5年,6年,7年,8年,9年,10年,15年,20年,25年,30年,40年
基準日,,,,,,,,,,,,,,,
1974-09-24,10.327,9.362,8.830,8.515,8.348,8.290,8.240,8.121,8.127,NaN,NaN,NaN,NaN,NaN,NaN
1974-09-25,10.333,9.364,8.831,8.516,8.348,8.290,8.240,8.121,8.127,NaN,NaN,NaN,NaN,NaN,NaN
1974-09-26,10.340,9.366,8.832,8.516,8.348,8.290,8.240,8.122,8.128,NaN,NaN,NaN,NaN,NaN,NaN
1974-09-27,10.347,9.367,8.833,8.517,8.349,8.290,8.240,8.122,8.128,NaN,NaN,NaN,NaN,NaN,NaN
1974-09-28,10.354,9.369,8.834,8.518,8.349,8.291,8.240,8.122,8.129,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-03-19,1.021,1.277,1.399,1.567,1.691,1.795,1.903,2.034,2.157,2.273,2.793,3.146,3.451,3.491,3.583
2026-03-23,1.041,1.311,1.437,1.611,1.746,1.848,1.953,2.079,2.198,2.322,2.830,3.188,3.495,3.522,3.606
2026-03-24,1.053,1.311,1.431,1.602,1.731,1.830,1.932,2.053,2.171,2.282,2.802,3.161,3.478,3.509,3.598


In [7]:
interest_rate_annual = interest_rate.resample("YE").last()
cond_yield = interest_rate_annual.index >= '2007-01-01'
cond_yield &= interest_rate_annual.index <= '2025-01-01'
yield_data = (interest_rate_annual[cond_yield]
    .stack()
    .reset_index()
    .rename(columns={"level_1": "maturity", 0: "rate"})
    .assign(**{
        "年 ": lambda df: df["基準日"].dt.strftime("%Y"),
        "年限 ": lambda df: df["maturity"].str.replace("年", "").astype(int),
        "金利 ": lambda df: df["rate"].astype(float),
    })
)
yield_data.tail()

,基準日,maturity,rate,年,年限,金利
265,2024-12-31,15年,1.553,2024,15,1.553
266,2024-12-31,20年,1.890,2024,20,1.890
267,2024-12-31,25年,2.081,2024,25,2.081
268,2024-12-31,30年,2.252,2024,30,2.252
269,2024-12-31,40年,2.556,2024,40,2.556


In [8]:
colers = px.colors.sequential.YlGn[::-1] + px.colors.sequential.Reds
yield_flg = px.line(yield_data,
    x="年限 ",
    y="金利 ",
    color="年 ",
    color_discrete_sequence=colers,
    markers=True,
)
yield_flg.update_layout(width=900, height=700, font={"size": 18},
    xaxis = {"ticksuffix": "年"},
    yaxis = {"ticksuffix": "%"},
)
yield_flg.show()

In [9]:
jp_rate = pd.read_csv(
    data_path / "fm02_m_1.csv",
    encoding="utf-8",
    skiprows=2,
    names=["date", "無担保コールレート O/N 月平均金利"],
    dtype={"date": str}
)
jp_rate["date"] = (
    pd.to_datetime(jp_rate["date"], format="%Y/%m", errors="coerce")
    .dt.to_period("M")
    .dt.to_timestamp()
)
jp_rate = jp_rate.set_index("date")
jp_rate.head()

,無担保コールレート O/N 月平均金利
date,
1970-01-01,NaN
1970-02-01,NaN
1970-03-01,NaN
1970-04-01,NaN
1970-05-01,NaN


In [10]:
fred_url = "https://api.stlouisfed.org/fred/series/observations"
series_mapping = {
    "DFF": "実効FF金利",
    "DFEDTARU": "上限金利",
    "DFEDTARL": "下限金利",
}
observation_start = "2007-01-01"
observation_end = "2024-12-31"
base_fred_params = {
    "api_key": FRED_API_KEY,
    "file_type": "json",
    "observation_start": observation_start,
    "observation_end": observation_end,
    "frequency": "m",
    "aggregation_method": "eop",
}
fred_dfs = []
for series_id, series_name in series_mapping.items():
    fred_params = base_fred_params.copy()
    fred_params["series_id"] = series_id
    fred_res = requests.get(fred_url, params=fred_params)
    fred_df = pd.DataFrame(fred_res.json()["observations"]).assign(**{
        "date": lambda df: pd.to_datetime(df["date"]),
        "value": lambda df: pd.to_numeric(df["value"], errors="coerce"),
        "金利種類": series_name,
    })
    fred_dfs.append(fred_df)
    time.sleep(5)
    
ffrate = pd.concat(fred_dfs)
ffrate

    

,realtime_start,realtime_end,date,value,金利種類
0,2026-03-30,2026-03-30,2007-01-01,5.33,実効FF金利
1,2026-03-30,2026-03-30,2007-02-01,5.41,実効FF金利
2,2026-03-30,2026-03-30,2007-03-01,5.30,実効FF金利
3,2026-03-30,2026-03-30,2007-04-01,5.29,実効FF金利
4,2026-03-30,2026-03-30,2007-05-01,5.28,実効FF金利
...,...,...,...,...,...
188,2026-03-31,2026-03-31,2024-08-01,5.25,下限金利
189,2026-03-31,2026-03-31,2024-09-01,4.75,下限金利
190,2026-03-31,2026-03-31,2024-10-01,4.75,下限金利
191,2026-03-31,2026-03-31,2024-11-01,4.50,下限金利


In [11]:
ff_fig = px.line(ffrate, x="date", y="value", color="金利種類")
ff_fig.update_layout(width=900, height=450, font={"size": 18},
    xaxis = {"ticksuffix": "年"},
    yaxis = {"ticksuffix": "%"},
)
ff_fig.show()

In [12]:
fedfunds_params = base_fred_params.copy()
fedfunds_params["series_id"] = "FEDFUNDS"
fedfunds_params["observation_start"] = "1985-01-01"
fedfunds_res = requests.get(fred_url, params=fedfunds_params)
print(fedfunds_res.json())
fedfunds_df = pd.DataFrame(fedfunds_res.json()["observations"]).assign(**{
    "date": lambda df: pd.to_datetime(df["date"]),
    "米国": lambda df: pd.to_numeric(df["value"], errors="coerce"),
})[["date", "米国"]]
fedfunds_df

{'realtime_start': '2026-03-27', 'realtime_end': '2026-03-27', 'observation_start': '1985-01-01', 'observation_end': '2024-12-31', 'units': 'lin', 'output_type': 1, 'file_type': 'json', 'order_by': 'observation_date', 'sort_order': 'asc', 'count': 480, 'offset': 0, 'limit': 100000, 'observations': [{'realtime_start': '2026-03-27', 'realtime_end': '2026-03-27', 'date': '1985-01-01', 'value': '8.35'}, {'realtime_start': '2026-03-27', 'realtime_end': '2026-03-27', 'date': '1985-02-01', 'value': '8.50'}, {'realtime_start': '2026-03-27', 'realtime_end': '2026-03-27', 'date': '1985-03-01', 'value': '8.58'}, {'realtime_start': '2026-03-27', 'realtime_end': '2026-03-27', 'date': '1985-04-01', 'value': '8.27'}, {'realtime_start': '2026-03-27', 'realtime_end': '2026-03-27', 'date': '1985-05-01', 'value': '7.97'}, {'realtime_start': '2026-03-27', 'realtime_end': '2026-03-27', 'date': '1985-06-01', 'value': '7.53'}, {'realtime_start': '2026-03-27', 'realtime_end': '2026-03-27', 'date': '1985-07-01

,date,米国
0,1985-01-01,8.35
1,1985-02-01,8.50
2,1985-03-01,8.58
3,1985-04-01,8.27
4,1985-05-01,7.97
...,...,...
475,2024-08-01,5.33
476,2024-09-01,5.13
477,2024-10-01,4.83
478,2024-11-01,4.64


In [13]:
rate_merged = fedfunds_df.merge(
    jp_rate, left_on="date", right_index=True, how="left"
).rename(columns={"無担保コールレート O/N 月平均金利": "日本"})
rate_tidy = pd.melt(rate_merged, 
    id_vars="date", 
    value_vars=["米国", "日本"],
    var_name="国", 
    value_name="利率",
    ignore_index=False,
).reset_index()
rate_fig = px.line(rate_tidy, x="date", y="利率", color="国")
rate_fig.update_layout(width=900, height=450, font={"size": 18},
    xaxis = {"ticksuffix": "年", "title": None},
    yaxis = {"ticksuffix": "%"},
)
rate_fig.show()


In [14]:
fx_params = base_fred_params.copy()
fx_params["series_id"] = "DEXJPUS"
fx_params["observation_start"] = "1985-01-01"
fx_params["observation_end"] = "2024-12-31"
fx_res = requests.get(fred_url, params=fx_params)
fx_df = pd.DataFrame(fx_res.json()["observations"]).assign(**{
    "date": lambda df: pd.to_datetime(df["date"]),
    "value": lambda df: pd.to_numeric(df["value"], errors="coerce")
})[["date", "value"]]

In [15]:
rate_wide = rate_merged.assign(**{"金利差": rate_merged["米国"] - rate_merged["日本"]})
parity_fig = go.Figure()

parity_fig.add_trace(
    go.Scatter(x=fx_df["date"], y=fx_df["value"], name="USD/JPY", marker_color="blue")
)

parity_fig.add_trace(go.Scatter(
    x=rate_wide["date"], 
    y=rate_wide["金利差"], 
    name="金利差", 
    marker_color="red",
    yaxis="y2"
))
parity_fig.update_layout(width=900, height=450, font={"size": 18},
    xaxis = {"ticksuffix": "年", "title": None},
    yaxis = {"title": "為替レート(円)"},
    yaxis2 = {"title": "金利差(%)", "overlaying": "y", "side": "right"},
)
parity_fig.show()


In [16]:
start_period = "2015-01-01"
end_period = "2024-12-31"

frequency = "M"

currency_list = ["US", "DE", "JP", "GB", "AU", "CA"]

collection_indicators = "E"

currency = "+" . join(currency_list)

bis_base_url = "https://stats.bis.org/api/v2/data/dataflow/BIS/WS_XRU/1.0/"
bis_endpoint = f"{frequency}.{currency}..{collection_indicators}"
bis_endpoint_url = urljoin(bis_base_url, bis_endpoint)
bis_params = {"startPeriod": start_period, "endPeriod": end_period, "format": "csv"}
bis_request_url = urljoin(bis_endpoint_url, "?" + urlencode(bis_params))
use_cols = ["FREQ", "REF_AREA", "CURRENCY", "COLLECTION", "TIME_PERIOD", "OBS_VALUE"]
bis_fx_df = pd.read_csv(
    bis_request_url, usecols=use_cols, parse_dates=["TIME_PERIOD"], date_format="%Y-%m"
)
bis_fx_df

,FREQ,REF_AREA,CURRENCY,COLLECTION,TIME_PERIOD,OBS_VALUE
0,M,CA,CAD,E,2015-01-01,1.266962
1,M,CA,CAD,E,2015-02-01,1.245107
2,M,CA,CAD,E,2015-03-01,1.276884
3,M,CA,CAD,E,2015-04-01,1.201962
4,M,CA,CAD,E,2015-05-01,1.244303
...,...,...,...,...,...,...
715,M,AU,AUD,E,2024-08-01,1.470281
716,M,AU,AUD,E,2024-09-01,1.443909
717,M,AU,AUD,E,2024-10-01,1.521687
718,M,AU,AUD,E,2024-11-01,1.538061


In [17]:
fx_wide = (bis_fx_df[["TIME_PERIOD", "CURRENCY", "OBS_VALUE"]]
    .set_index(["TIME_PERIOD", "CURRENCY"])
    .unstack()
)
fx_change = fx_wide / fx_wide.iloc[0, :]
fx_tidy = (fx_change
    .stack()
    .reset_index()
    .rename(columns={"CURRENCY": "通貨", "OBS_VALUE": "実効為替レート"})
)
fx_fig = px.line(
    fx_tidy, x="TIME_PERIOD", y="実効為替レート", color="通貨", width=900, height=450,
)
fx_fig.update_layout(font={"size": 18},
    xaxis = {"ticksuffix": "年", "title": None},
)
fx_fig.show()

/tmp/ipykernel_80/3654216177.py:7: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  .stack()


### 9.3 マネーストックと資金循環

In [18]:
# open_data/data/md02_m_1.csv
money_boj = pd.read_csv(data_path / "md02_m_1.csv", encoding="utf-8", header=3)
money_boj.iloc[:10, [0] + list(range(7, 11))]

,系列名称,＿準通貨／平残前年比／マネーストック（2004年3月以前はマネーサプライ）,＿ＣＤ／平残前年比／マネーストック（2004年3月以前はマネーサプライ）,Ｍ２／平／マネーストック,Ｍ３／平／マネーストック
0,データコード,MD02'MAM1YAM3QMMO,MD02'MAM1YAM3CDMO,MD02'MAM1NAM2M2MO,MD02'MAM1NAM3M3MO
1,単位,％,％,億円,億円
2,収録開始期,1968/01,1980/05,2003/04,2003/04
3,収録終了期,2026/02,2026/02,2026/02,2026/02
4,最終更新日,2026/03/10,2026/03/10,2026/03/10,2026/03/10
5,1980/01,11.7,NaN,NaN,NaN
6,1980/02,11.6,NaN,NaN,NaN
7,1980/03,11,NaN,NaN,NaN
8,1980/04,11.1,NaN,NaN,NaN
9,1980/05,12.4,1022.8,NaN,NaN


In [19]:
bond_boj = (
    bond_boj.rename(columns=lambda col: str(col).strip())
    .rename(
        columns={
            "＿種類別内訳/国債/内国債": "国債残高",
            "__種類別内訳＿種類別内訳/国債/内国債": "国債残高",
        }
    )
)

moneystock = money_boj.loc[5:, :].copy()
moneystock.columns = moneystock.columns.str.strip()
moneystock = moneystock.assign(
    年月=pd.to_datetime(moneystock["系列名称"], format="%Y/%m", errors="coerce"),
    年=pd.to_numeric(moneystock["系列名称"].str[:4], errors="coerce"),
    マネーストック=pd.to_numeric(moneystock["Ｍ２／平／マネーストック"], errors="coerce"),
)

# 年末かつ2005年以降に絞る
cond_m2 = (moneystock["年月"].dt.month == 12) & (moneystock["年"] > 2004)
m2 = moneystock.loc[cond_m2, ["年", "マネーストック"]]

# 国債残高と結合
bond_m2 = bond_boj.merge(m2, on="年", how="inner")[["年", "国債残高", "マネーストック"]]
bond_m2.tail()

,年,国債残高,マネーストック
15,2020,9838777,11359680.0
16,2021,10237117,11782086.0
17,2022,10626907,12128301.0
18,2023,10975593,12409742.0
19,2024,11325480,12576036.0


In [20]:
# 生前データ形式に変換
m2_tidy = pd.melt(bond_m2,
    id_vars="年",
    value_vars=["国債残高", "マネーストック"],
    var_name="残高",
    value_name="金額( 兆円 )",
)
# 兆円単位に変換
m2_tidy = m2_tidy.assign(**{"金額( 兆円 )": m2_tidy["金額( 兆円 )"] / 10000})
# 棒グラフで可視化
m2_flg = px.bar(m2_tidy, x="年", y="金額( 兆円 )", color="残高", barmode="group")
m2_flg.update_layout(width=900, height=450, font={"size": 18},
    xaxis = {"ticksuffix": "年", "title": None},
    yaxis = {"tickformat": ".0f", "ticksuffix": "兆円", "title": None},
)
m2_flg.show()

In [21]:
m2_params = base_fred_params.copy()
m2_params["observation_start"] = "1990-01-01"
m2_params["observation_end"] = "2024-12-31"
m2_params["frequency"] = "a"
m2_series_mapping = {
    "M2SL": "M2( マネーストック )",
    "GDP": " 名目 GDP"
}
m2_fred_dfs = []
for series_id, series_name in m2_series_mapping.items():
    m2_fred_params = m2_params.copy()
    m2_fred_params["series_id"] = series_id
    fred_res = requests.get(fred_url, params=m2_fred_params)
    fred_df = pd.DataFrame(fred_res.json()["observations"]).assign(**{
        " 日付 ": lambda df: pd.to_datetime(df["date"]),
        "value": lambda df: pd.to_numeric(df["value"], errors="coerce"),
        " 系列名 ": series_name,
    })
    m2_fred_dfs.append(fred_df)
m2_fred_df = pd.concat(m2_fred_dfs)
m2_gdp_pivot = m2_fred_df.pivot_table(values="value", index=" 日付 ", columns=" 系列名 ")
m2_gdp = m2_gdp_pivot.assign(**{
    "m2velocity": m2_gdp_pivot[" 名目 GDP"] /  m2_gdp_pivot["M2( マネーストック )"],
    "marsshall_k": m2_gdp_pivot["M2( マネーストック )"] / m2_gdp_pivot[" 名目 GDP"],
})
m2_gdp.tail()

系列名,名目 GDP,M2( マネーストック ),m2velocity,marsshall_k
日付,,,,
2020-01-01,22087.160,19115.7,1.155446,0.865467
2021-01-01,24813.600,21498.5,1.154201,0.866400
2022-01-01,26770.514,21290.9,1.257369,0.795312
2023-01-01,28424.722,20777.8,1.368033,0.730976
2024-01-01,29825.182,21485.3,1.388167,0.720374


In [22]:
m2_gdp_tidy = m2_fred_df[[" 日付 ", " 系列名 ", "value"]]
m2_gdp_flg = px.line(m2_gdp_tidy, x=" 日付 ", y="value", color=" 系列名 ")
m2_gdp_flg.update_layout(width=900, height=450, font={"size": 18},
    xaxis = {"ticksuffix": "年", "title": None},
    yaxis = {"tickformat": ",.0f", "ticksuffix": "十億ドル", "title": None},
)
m2_gdp_flg.show()

In [30]:
cols_boj = [
    "系列名称",
    "負債・資金過不足／中央銀行／フロー",
    "負債・資金過不足／預金取扱機関／フロー",
    "負債・資金過不足／民間非金融法人企業／フロー",
    "負債・資金過不足／一般政府／フロー",
    "負債・資金過不足／うち公的年金／フロー",
    "負債・資金過不足／家計／フロー",
    "負債・資金過不足／海外／フロー",
]
fund_flow_df = pd.read_csv(
    data_path / "nme_資金循環_資金過不足_年度.csv",
    header=1,
    encoding="utf-8",
    usecols=cols_boj,
)
fund_flow_df.head(3).T

,0,1,2
系列名称,2008,2009,2010
負債・資金過不足／中央銀行／フロー,-7,3437,2228
負債・資金過不足／預金取扱機関／フロー,79759,107821,38533
負債・資金過不足／民間非金融法人企業／フロー,76904,287993,320981
負債・資金過不足／一般政府／フロー,-282910,-444220,-401919
負債・資金過不足／うち公的年金／フロー,-44440,-31415,-35277
負債・資金過不足／家計／フロー,118975,137422,199975
負債・資金過不足／海外／フロー,-101945,-162664,-177883


In [33]:
fund_flow_tidy = (fund_flow_df
    .set_index("系列名称")
    .stack()
    .reset_index()
    .rename(columns={"level_1": " 部門 ", 0: " 資金過不足 "})
)
fund_flow_tidy = fund_flow_tidy.assign(**{
    " 部門 ": lambda df: df[' 部門 '].str.split("／", expand=True).iloc[:, 1]
})
fund_flow_fig = px.line(
    fund_flow_tidy, x="系列名称", y=" 資金過不足 ", color=" 部門 ", title=" 資金過不足 "
)
fund_flow_fig.update_layout(width=900, height=450, font={"size": 18},
    xaxis = {"ticksuffix": "年", "title": None},
    yaxis = {"tickformat": ",.0f", "ticksuffix": "億円", "title": None},
)
fund_flow_fig.show()